## Exposure Time Calculator for SDSU MLO Spectrograph

In this notebook, we will develop a rudimentary exposure time calculator that either a) given an exposure time, outputs the SNR or b) given a SNR, outputs the exposure time.

In [2]:
#import libs:
import numpy as np


### Basic setup:

Use equation: $$m - m_{zp} = -2.5 \log_{10}([\mathrm{counts}])$$ to get counts.

For SNR: $$SNR = \frac{S_{obj}}{\sqrt{S_{obj} + S_{sky} + S_{dark} + R^2}}$$

This translates to the CCD equation: $$SNR = \frac{S_{obj}\sqrt{Q_e t} }{\sqrt{S_{obj} + n_{pix}(S_{sky} + (\frac{S_{dark}}{Q_e}) + (\frac{R^2}{Q_e t}))}}$$

$Q_e$: quantum efficiency

$S_{obj}$: object counts (phot/sec)

$S_{sky}$: sky counts (background??) (phot/sec/pix)

$S_{dark}$: dark current (elec/sec/pix)

$n_{pix}$: pixel size of object

$R$: readout noise (elec/pix)

$t$: **exposure time**

*probably use algebra to reverse this equation?*

In [19]:
def get_counts_from_mag(mag, mag_zp):
    """Given an apparent magnitude and zero-point magnitude, output the number of counts asscociated with the magnitude."""
    counts = np.power(10, (mag_zp - mag)/2.5)

    return counts

def get_SNR(exp_time, mag, mag_zp, ob_size, sky_noise, dark_current, quant_eff, read_noise):
    """Predicts the SNR given an exposure time.
    
    Parameters:

    exp_time: exposure time (s)

    mag: apparent magnitude of object

    mag_zp: zero-point magnitude

    ob_size: size of object, in pixels

    sky_noise: background counts per pixel (phot/s/pix)

    dark_current: dark current (elec/s/pix)

    quant_eff: quantum efficiency (between 0 and 1)

    read_noise: time independent detector noise (elec/pix)
    """
    s_ob = get_counts_from_mag(mag, mag_zp)

    numerator = s_ob * np.sqrt(quant_eff * exp_time)
    
    denominator = np.sqrt(s_ob + ob_size * (sky_noise + (dark_current/quant_eff) + (read_noise**2/(quant_eff * exp_time))))

    snr = numerator/denominator
    
    return snr

In [ ]:
def get_exptime(SNR, mag, mag_zp, ob_size, sky_noise, dark_current, quant_eff, read_noise):
    """Predicts the best exposure time to achieve a desired SNR.
    
    Parameters:

    SNR: signal to noise ratio

    mag: apparent magnitude of object

    mag_zp: zero-point magnitude

    ob_size: size of object, in pixels

    sky_noise: background counts per pixel (phot/s/pix)

    dark_current: dark current (elec/s/pix)

    quant_eff: quantum efficiency (between 0 and 1)

    read_noise: time independent detector noise (elec/pix)
    """

    s_ob = get_counts_from_mag(mag, mag_zp)

    C = s_ob/ob_size + sky_noise + dark_current/quant_eff

    exp_time = (C * ob_size * SNR**2 * quant_eff + np.sqrt((SNR**2 * quant_eff * ob_size * C)**2 + 4 * (s_ob * quant_eff)**2 * (read_noise**2 * ob_size * SNR**2)))/(2 * (s_ob * quant_eff)**2)

    return exp_time

In [29]:
#create some mock values to test:

mag = 16
mag_zp = 24
exp_time = 30.0

pix_rad = 5
ob_size = np.pi * pix_rad**2

dark_current = 0.01
sky_noise = 5.0
quant_eff = 0.75
read_noise = 5.0

get_SNR(exp_time, mag, mag_zp, ob_size, sky_noise, dark_current, quant_eff, read_noise)

np.float64(165.4001836679808)